# Visual HMTL V4 训练

**目标**: 提升非激活消极类准确率

**优化策略**:
1. 调整类别权重：非激活消极 ×1.5，平静 ×0.8
2. Label Smoothing = 0.1
3. 增加 A/V 损失权重，强化激活度学习

## 1. 环境检查

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. 挂载 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_PATH = "/content/drive/MyDrive"
ZIP_FILE = f"{DRIVE_PATH}/archive.zip"
LABELS_CSV = f"{DRIVE_PATH}/visual_hmtl_labels.csv"
MODEL_SAVE_PATH = f"{DRIVE_PATH}/visual_hmtl_v4_best.pt"

print(f"压缩包存在: {os.path.exists(ZIP_FILE)}")
print(f"标签文件存在: {os.path.exists(LABELS_CSV)}")

In [ ]:
import zipfile

LOCAL_DATA_DIR = "/content/visual_data"

if not os.path.exists(LOCAL_DATA_DIR):
    print(f"解压 {ZIP_FILE}...")
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_DATA_DIR)
    print("解压完成!")
else:
    print("数据已存在")

import glob
train_dirs = glob.glob(f"{LOCAL_DATA_DIR}/**/Train", recursive=True)
test_dirs = glob.glob(f"{LOCAL_DATA_DIR}/**/Test", recursive=True)

TRAIN_DIR = train_dirs[0] if train_dirs else None
TEST_DIR = test_dirs[0] if test_dirs else None

print(f"Train目录: {TRAIN_DIR}")
print(f"Test目录: {TEST_DIR}")

## 3. 定义模型

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models


class FocalLossWithSmoothing(nn.Module):
    """Focal Loss + Label Smoothing"""
    def __init__(self, alpha=None, gamma=2.0, smoothing=0.1, num_classes=4):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing
        self.num_classes = num_classes
    
    def forward(self, inputs, targets):
        # Label smoothing
        with torch.no_grad():
            smooth_targets = torch.zeros_like(inputs)
            smooth_targets.fill_(self.smoothing / (self.num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        
        # Log softmax
        log_probs = F.log_softmax(inputs, dim=1)
        
        # Cross entropy with smoothing
        ce_loss = -(smooth_targets * log_probs).sum(dim=1)
        
        # Focal weight
        probs = torch.exp(log_probs)
        pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_weight = (1 - pt) ** self.gamma
        
        # Alpha weight
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_weight = alpha_t * focal_weight
        
        loss = focal_weight * ce_loss
        return loss.mean()


class VisualHMTLClassifierV4(nn.Module):
    """V4: 增强激活度学习"""
    
    def __init__(self, dropout=0.3):
        super().__init__()
        
        self.backbone = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)
        feature_dim = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Identity()
        
        # 共享层
        self.shared_fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feature_dim, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout * 0.5)
        )
        
        # 4类分类头 (主要)
        self.classifier_4 = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, 4)
        )
        
        # 3类分类头
        self.classifier_3 = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, 3)
        )
        
        # 7类分类头
        self.classifier_7 = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, 7)
        )
        
        # Arousal (更大的网络，强化学习)
        self.regressor_A = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
        
        # Valence (更大的网络)
        self.regressor_V = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Tanh()
        )
    
    def forward(self, x):
        features = self.backbone(x)
        shared = self.shared_fc(features)
        
        return {
            'label_4_logits': self.classifier_4(shared),
            'label_3_logits': self.classifier_3(shared),
            'label_7_logits': self.classifier_7(shared),
            'arousal': self.regressor_A(shared).squeeze(-1),
            'valence': self.regressor_V(shared).squeeze(-1)
        }


print("模型定义完成!")

## 4. 数据集和损失函数

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd


class VisualEmotionDatasetV4(Dataset):
    def __init__(self, df, train_dir, test_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.train_dir = train_dir
        self.test_dir = test_dir
        self.transform = transform
        
        self.valid_indices = []
        for idx in range(len(self.df)):
            img_path = self._get_image_path(idx)
            if img_path and os.path.exists(img_path):
                self.valid_indices.append(idx)
        
        print(f"有效样本: {len(self.valid_indices)}/{len(self.df)}")
    
    def _get_image_path(self, idx):
        row = self.df.iloc[idx]
        image_path = row['image_path']
        original_path = row['original_path']
        
        if 'Train' in image_path:
            base_dir = self.train_dir
        elif 'Test' in image_path:
            base_dir = self.test_dir
        else:
            return None
        
        if base_dir is None:
            return None
        
        return os.path.join(base_dir, original_path)
    
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, idx):
        real_idx = self.valid_indices[idx]
        row = self.df.iloc[real_idx]
        img_path = self._get_image_path(real_idx)
        
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
        except:
            image = torch.zeros(3, 224, 224)
        
        return {
            'image': image,
            'label_7': torch.tensor(row['label_7'], dtype=torch.long),
            'label_4': torch.tensor(row['label_4'], dtype=torch.long),
            'label_3': torch.tensor(row['label_3'], dtype=torch.long),
            'arousal': torch.tensor(row['arousal'], dtype=torch.float),
            'valence': torch.tensor(row['valence'], dtype=torch.float),
        }


class HMTLVisualLossV4(nn.Module):
    """V4 Loss: 强化激活度学习"""
    def __init__(self, class_weights_4=None):
        super().__init__()
        # 4类: Focal Loss + Label Smoothing
        self.loss_4 = FocalLossWithSmoothing(
            alpha=class_weights_4, 
            gamma=2.0, 
            smoothing=0.1,
            num_classes=4
        )
        self.loss_3 = nn.CrossEntropyLoss(label_smoothing=0.1)
        self.loss_7 = nn.CrossEntropyLoss()
        self.mse_loss = nn.MSELoss()
    
    def forward(self, outputs, targets):
        l4 = self.loss_4(outputs['label_4_logits'], targets['label_4'])
        l3 = self.loss_3(outputs['label_3_logits'], targets['label_3'])
        l7 = self.loss_7(outputs['label_7_logits'], targets['label_7'])
        l_a = self.mse_loss(outputs['arousal'], targets['arousal'])
        l_v = self.mse_loss(outputs['valence'], targets['valence'])
        
        # V4: 增加 A/V 权重，强化激活度学习
        # 4类权重高，A/V权重也高
        total = 1.5 * l4 + 0.8 * l3 + 0.5 * l7 + 1.0 * l_a + 0.8 * l_v
        
        return {
            'total_loss': total,
            'loss_4': l4.item(),
            'loss_3': l3.item(),
            'loss_7': l7.item(),
            'loss_a': l_a.item(),
            'loss_v': l_v.item(),
        }


# 数据增强 (更强)
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1)
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("数据集和损失函数定义完成!")

## 5. 加载数据

In [ ]:
print(f"加载标签: {LABELS_CSV}")
label_df = pd.read_csv(LABELS_CSV)
print(f"总样本数: {len(label_df)}")

train_df = label_df[label_df['image_path'].str.contains('Train')].copy()
test_df = label_df[label_df['image_path'].str.contains('Test')].copy()

print(f"Train样本: {len(train_df)}")
print(f"Test样本: {len(test_df)}")

LABEL_4_NAMES = {0: '积极', 1: '激活消极', 2: '非激活消极', 3: '平静'}
print("\nTrain 4类分布:")
for label_id in sorted(train_df['label_4'].unique()):
    count = len(train_df[train_df['label_4'] == label_id])
    print(f"  [{label_id}] {LABEL_4_NAMES.get(label_id, '未知')}: {count}")

In [ ]:
BATCH_SIZE = 32

print("创建训练集...")
train_dataset = VisualEmotionDatasetV4(train_df, TRAIN_DIR, TEST_DIR, transform=train_transform)

print("\n创建测试集...")
test_dataset = VisualEmotionDatasetV4(test_df, TRAIN_DIR, TEST_DIR, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\n训练批次: {len(train_loader)}, 测试批次: {len(test_loader)}")

## 6. 创建模型

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"设备: {device}")

model = VisualHMTLClassifierV4().to(device)

# V4 类别权重: 非激活消极 ×1.5, 平静 ×0.8
label_4_counts = train_df['label_4'].value_counts().to_dict()
total = len(train_df)
CLASS_WEIGHTS_4 = torch.tensor([
    total / (4 * label_4_counts.get(0, 1)),        # 积极
    total / (4 * label_4_counts.get(1, 1)),        # 激活消极
    total / (4 * label_4_counts.get(2, 1)) * 1.5,  # 非激活消极 ×1.5
    total / (4 * label_4_counts.get(3, 1)) * 0.8,  # 平静 ×0.8
]).to(device)
print(f"4类权重: {CLASS_WEIGHTS_4}")

criterion = HMTLVisualLossV4(class_weights_4=CLASS_WEIGHTS_4)

LEARNING_RATE = 1e-4

backbone_params = list(model.backbone.parameters())
head_params = (
    list(model.shared_fc.parameters()) +
    list(model.classifier_4.parameters()) +
    list(model.classifier_3.parameters()) +
    list(model.classifier_7.parameters()) +
    list(model.regressor_A.parameters()) +
    list(model.regressor_V.parameters())
)

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': LEARNING_RATE * 0.1},
    {'params': head_params, 'lr': LEARNING_RATE}
], weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)

print("模型创建完成!")

## 7. 训练

In [ ]:
from tqdm import tqdm

def evaluate(model, dataloader, device):
    model.eval()
    correct_4, correct_3, correct_7 = 0, 0, 0
    total = 0
    
    class_correct_4 = {i: 0 for i in range(4)}
    class_total_4 = {i: 0 for i in range(4)}
    
    with torch.no_grad():
        for batch in dataloader:
            images = batch['image'].to(device)
            targets = {k: v.to(device) for k, v in batch.items() if k != 'image'}
            
            outputs = model(images)
            
            pred_4 = outputs['label_4_logits'].argmax(dim=1)
            pred_3 = outputs['label_3_logits'].argmax(dim=1)
            pred_7 = outputs['label_7_logits'].argmax(dim=1)
            
            correct_4 += (pred_4 == targets['label_4']).sum().item()
            correct_3 += (pred_3 == targets['label_3']).sum().item()
            correct_7 += (pred_7 == targets['label_7']).sum().item()
            total += targets['label_4'].size(0)
            
            for i in range(len(pred_4)):
                t4 = targets['label_4'][i].item()
                class_total_4[t4] += 1
                if pred_4[i].item() == t4:
                    class_correct_4[t4] += 1
    
    return {
        'acc_4': correct_4 / total if total > 0 else 0,
        'acc_3': correct_3 / total if total > 0 else 0,
        'acc_7': correct_7 / total if total > 0 else 0,
        'class_acc_4': {k: class_correct_4[k]/class_total_4[k] if class_total_4[k] > 0 else 0 for k in range(4)}
    }


NUM_EPOCHS = 20
best_score = 0  # 综合分数

print("="*60)
print("开始训练 Visual HMTL V4")
print("优化目标: 提升非激活消极类准确率")
print(f"Epochs: {NUM_EPOCHS}, Batch Size: {BATCH_SIZE}")
print("="*60)

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for batch in pbar:
        images = batch['image'].to(device)
        targets = {k: v.to(device) for k, v in batch.items() if k != 'image'}
        
        optimizer.zero_grad()
        outputs = model(images)
        loss_dict = criterion(outputs, targets)
        
        loss_dict['total_loss'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        epoch_loss += loss_dict['total_loss'].item()
        pbar.set_postfix({'loss': f"{loss_dict['total_loss'].item():.4f}"})
    
    scheduler.step()
    
    # 评估
    eval_result = evaluate(model, test_loader, device)
    
    # 综合分数: 4类整体 + 非激活消极类准确率
    score = eval_result['acc_4'] * 0.6 + eval_result['class_acc_4'][2] * 0.4
    
    print(f"\nEpoch {epoch+1}: Loss={epoch_loss/len(train_loader):.4f}")
    print(f"  4类: {eval_result['acc_4']*100:.2f}%, 3类: {eval_result['acc_3']*100:.2f}%, 7类: {eval_result['acc_7']*100:.2f}%")
    print(f"  4类每类: 积极={eval_result['class_acc_4'][0]*100:.1f}%, "
          f"激活消极={eval_result['class_acc_4'][1]*100:.1f}%, "
          f"非激活消极={eval_result['class_acc_4'][2]*100:.1f}%, "
          f"平静={eval_result['class_acc_4'][3]*100:.1f}%")
    print(f"  综合分数: {score*100:.2f}%")
    
    if score > best_score:
        best_score = score
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch,
            'acc_4': eval_result['acc_4'],
            'acc_3': eval_result['acc_3'],
            'acc_7': eval_result['acc_7'],
            'class_acc_4': eval_result['class_acc_4'],
            'score': score,
        }, MODEL_SAVE_PATH)
        print(f"  -> 保存最佳模型!")

print("\n" + "="*60)
print(f"训练完成! 最佳综合分数: {best_score*100:.2f}%")
print(f"模型保存到: {MODEL_SAVE_PATH}")
print("="*60)

## 8. 测试

In [ ]:
checkpoint = torch.load(MODEL_SAVE_PATH)
print(f"最佳模型 (Epoch {checkpoint['epoch']+1}):")
print(f"  4类: {checkpoint['acc_4']*100:.2f}%")
print(f"  3类: {checkpoint['acc_3']*100:.2f}%")
print(f"  7类: {checkpoint['acc_7']*100:.2f}%")
print(f"\n4类每类:")
for k, v in checkpoint['class_acc_4'].items():
    print(f"  {LABEL_4_NAMES[k]}: {v*100:.1f}%")

## 9. 下载模型

In [ ]:
from google.colab import files
files.download(MODEL_SAVE_PATH)
print("下载完成!")